In [13]:
#importing libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [21]:
#importing the data to a DataFrame

df= pd.read_csv("data/fmcg_sales_3years_1M_rows.csv", parse_dates=["date"])

In [23]:
#checking first 5 rows of data to confirm data loaded

df.head()

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,2021-01-01,2021,1,1,53,4,0,1,8.44,1.24,...,0.1,1,167.84,151.06,248,0,11,S008,7.53,0.182
1,2021-01-02,2021,1,2,53,5,1,0,12.61,1.12,...,0.0,0,125.88,125.88,238,0,6,S057,5.19,0.505
2,2021-01-03,2021,1,3,53,6,1,0,12.02,2.69,...,0.3,1,398.62,279.03,238,0,6,S017,5.59,0.168
3,2021-01-04,2021,1,4,1,0,0,0,7.76,4.65,...,0.0,0,83.92,83.92,216,0,7,S012,7.81,0.255
4,2021-01-05,2021,1,5,1,1,0,0,11.16,1.77,...,0.2,1,178.33,142.66,372,0,8,S038,7.62,0.073


In [16]:
#checking rows and columns of the data frame

df.shape
print(f"Number of rows: {df.shape[0]}, Number of columns: {df.shape[1]}")

Number of rows: 1100000, Number of columns: 33


In [24]:
#checking column names, data types for the rows

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100000 entries, 0 to 1099999
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            1100000 non-null  datetime64[ns]
 1   year            1100000 non-null  int64         
 2   month           1100000 non-null  int64         
 3   day             1100000 non-null  int64         
 4   weekofyear      1100000 non-null  int64         
 5   weekday         1100000 non-null  int64         
 6   is_weekend      1100000 non-null  int64         
 7   is_holiday      1100000 non-null  int64         
 8   temperature     1100000 non-null  float64       
 9   rain_mm         1100000 non-null  float64       
 10  store_id        1100000 non-null  object        
 11  country         1100000 non-null  object        
 12  city            1100000 non-null  object        
 13  channel         1100000 non-null  object        
 14  latitude        11

In [39]:
#checking missing values in each row

df.isna().sum()

date              0
year              0
month             0
day               0
weekofyear        0
weekday           0
is_weekend        0
is_holiday        0
temperature       0
rain_mm           0
store_id          0
country           0
city              0
channel           0
latitude          0
longitude         0
sku_id            0
sku_name          0
category          0
subcategory       0
brand             0
units_sold        0
list_price        0
discount_pct      0
promo_flag        0
gross_sales       0
net_sales         0
stock_on_hand     0
stock_out_flag    0
lead_time_days    0
supplier_id       0
purchase_cost     0
margin_pct        0
dtype: int64

In [40]:
# standardising text labels
text_cols = ["country", "city", "channel", "category", "subcategory", "brand"]
for col in text_cols:
    before = df[col].nunique()
    df[col] = df[col].str.strip()             # remove stray spaces
    print(f"{col:12s} unique labels before/after strip: {before} -> {df[col].nunique()}")

# Always LOOK at the unique values - this is how you spot "E-comm" vs "E-commerce"
print("\nChannels:", sorted(df["channel"].unique()))
print("Countries:", sorted(df["country"].unique()))

country      unique labels before/after strip: 7 -> 7
city         unique labels before/after strip: 9 -> 9
channel      unique labels before/after strip: 4 -> 4
category     unique labels before/after strip: 5 -> 5
subcategory  unique labels before/after strip: 17 -> 17
brand        unique labels before/after strip: 6 -> 6

Channels: ['Convenience', 'E-commerce', 'Hypermarket', 'Supermarket']
Countries: ['Austria', 'France', 'Germany', 'Italy', 'Netherlands', 'Poland', 'Spain']


In [19]:
# how many unique values are present in each of the specified columns
df[["store_id","country","city","channel","brand"]].nunique()

store_id    13
country      7
city         9
channel      4
brand        6
dtype: int64

In [41]:
# checking for impossible values 
print("Negative units:", (df["units_sold"] < 0).sum())
print("Zero or negative price:", (df["list_price"] <= 0).sum())
print("Zero-unit days:", (df["units_sold"] == 0).sum(), f"({(df['units_sold'] == 0).mean():.2%})")
print("Negative margin rows:", (df["margin_pct"] < 0).sum(), "(kept - selling below cost is a business signal)")


Negative units: 0
Zero or negative price: 0
Zero-unit days: 3092 (0.28%)
Negative margin rows: 3570 (kept - selling below cost is a business signal)


In [25]:
# checking first and last dates in the file

print("First date:", df["date"].min())
print("Last date:", df["date"].max())
print("Number of SKUs:", df["sku_id"].nunique())

First date: 2021-01-01 00:00:00
Last date: 2023-12-31 00:00:00
Number of SKUs: 102


In [45]:
# Missing DAYS: does every store-SKU series have all 1,095 days?
days = df.groupby(["store_id", "sku_id"])["date"].agg(["min", "max", "count"])
short = days[days["count"] < 1095]
print("Stores with gaps:", len(short), "of", len(days))
short   # one store stops early = a product that was discontinued in that store

Stores with gaps: 1 of 1005


,,min,max,count
store_id,sku_id,,,
STORE0013,SKU0073,2021-01-01,2022-09-12,620


In [26]:
# checking that each row is unique per Date +Store Id +SKU ID + Channel
record_key = ["date", "store_id", "sku_id", "channel"]

rows_before_duplicates = len(df)
duplicate_count = df.duplicated(subset=record_key).sum()
print("Repeated rows:", duplicate_count)

df = df.drop_duplicates(subset=record_key, keep="first").copy()
rows_after_duplicates = len(df)
print("Rows before:", rows_before_duplicates)
print("Rows after:", rows_after_duplicates)

Repeated rows: 0
Rows before: 1100000
Rows after: 1100000


In [27]:
# daily demand for each sku 

daily_sku_sales = df.groupby(["date", "sku_id"], as_index=False).agg(
    units_sold=("units_sold", "sum"),
    net_sales=("net_sales", "sum"),
    holiday_market_share=("is_holiday", "mean")
)

display(daily_sku_sales.head())
print("Daily SKU rows and columns:", daily_sku_sales.shape)

,date,sku_id,units_sold,net_sales,holiday_market_share
0,2021-01-01,SKU0001,1118,6976.32,1.0
1,2021-01-01,SKU0002,643,7233.75,1.0
2,2021-01-01,SKU0003,163,518.34,1.0
3,2021-01-01,SKU0004,985,1782.85,1.0
4,2021-01-01,SKU0005,1694,11170.23,1.0


Daily SKU rows and columns: (111690, 5)


In [30]:
# compairing sales across channels 

channel_summary = df.groupby("channel", as_index=False).agg(
    average_units=("units_sold", "mean"),
    total_sales=("net_sales", "sum")
)
channel_summary.sort_values("total_sales", ascending=False).style.format({
        "average_units": "{:,.2f}",
        "total_sales": "{:,.2f}"
    })

,channel,average_units,total_sales
2,Hypermarket,67.24,"258,738,120.70"
1,E-commerce,59.54,"112,734,523.12"
3,Supermarket,47.17,"88,957,659.41"
0,Convenience,35.46,"12,516,149.60"


In [35]:
# checking product revenue

sku_revenue = df.groupby("sku_id").agg(
    total_revenue=("net_sales", "sum"),
    total_units=("units_sold", "sum")
)
sku_revenue = sku_revenue.sort_values("total_revenue", ascending=False)
sku_revenue.head(10)

,total_revenue,total_units
sku_id,,
SKU0018,17032424.64,1173032
SKU0027,16000295.02,1200897
SKU0026,13025419.73,893989
SKU0099,12968809.43,1242389
SKU0092,12496995.90,895842
SKU0100,12351150.68,920354
SKU0064,12326994.00,990120
SKU0061,11967260.40,972948
SKU0028,11749108.59,1253907


In [37]:
# compairing sales across countries 

country_summary = df.groupby("country").agg(
    average_units=("units_sold", "mean"),
    total_sales=("net_sales", "sum")
)
country_summary.sort_values("total_sales", ascending=False).style.format({
        "average_units": "{:,.2f}",
        "total_sales": "{:,.2f}"
    })

,average_units,total_sales
country,,
Italy,55.26,"137,193,076.12"
Spain,55.81,"106,704,767.04"
Germany,67.00,"88,525,559.01"
Austria,65.55,"43,020,025.97"
France,68.66,"42,601,612.57"
Poland,66.90,"42,385,262.52"
Netherlands,35.46,"12,516,149.60"


In [43]:
# promo_flag should be 1 exactly when there is a discount
promo_ok = ((df["promo_flag"] == 1) == (df["discount_pct"] > 0)).mean()
print(f"promo_flag correctly tagged with discount_pct on {promo_ok:.1%} of rows")

# net_sales should equal gross_sales * (1 - discount)
recon = (df["net_sales"] - df["gross_sales"] * (1 - df["discount_pct"])).abs() < 0.05
print(f"net_sales adds up on {recon.mean():.1%} of rows")

promo_flag correctly tagged with discount_pct on 100.0% of rows
net_sales adds up on 100.0% of rows


In [47]:
# stock out days 
avg_units = df.groupby("stock_out_flag")["units_sold"].mean()
stockout_rate = df["stock_out_flag"].mean()

print(f"Average units sold (In Stock):     {avg_units[0]:.1f} units/day")
print(f"Average units sold (Out of Stock): {avg_units[1]:.1f} units/day")
print(f"Percentage of days with a stockout: {stockout_rate:.1%}")

Average units sold (In Stock):     60.7 units/day
Average units sold (Out of Stock): 11.7 units/day
Percentage of days with a stockout: 3.0%
